# ISOM 835 · Session 9 — Trust, but Verify: Tuning, Explaining & Uncertainty
**Suffolk University · Sawyer Business School · Fall 2026 · Mon Nov 16 · Prof. Hasan Arslan**

Cross-validation that doesn't lie, hyperparameter search with Optuna, SHAP for one booking and the whole book, conformal intervals that keep their promise, fairness slices, and the one-page model card.

> **Frame it.** Same hotel-cancellation model as Session 8. Tonight's questions are the executive's: *why did it say that? how sure is it? is it fair? what do I watch after launch?*

In [ ]:
# Environment check — run this cell first. If it fails in Colab: run  !pip install -q -U scikit-learn pandas  then Runtime → Restart session.
import sys, re, sklearn, pandas as pd, numpy as np
need = {'scikit-learn': ('1.6', sklearn.__version__), 'pandas': ('2.2', pd.__version__), 'numpy': ('1.26', np.__version__)}
v = lambda s: tuple(int(x) for x in re.findall(r'\d+', s)[:2])
old = {k: have for k, (want, have) in need.items() if v(have) < v(want)}
assert not old, f'please upgrade {old}: !pip install -q -U ' + ' '.join(old)
print(f'Python {sys.version.split()[0]} ·', ' · '.join(f'{k} {have}' for k, (_, have) in need.items()), '✓')

In [ ]:
import pandas as pd, numpy as np, time
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, KFold, GroupKFold, StratifiedKFold, TimeSeriesSplit, cross_val_score, RandomizedSearchCV
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, confusion_matrix

URL = 'https://raw.githubusercontent.com/harslan/isom-835/master/public/data/hotel_bookings.csv'
raw = pd.read_csv(URL).sample(40_000, random_state=835)                  # subsample for speed in class
LEAKS = ['reservation_status', 'reservation_status_date', 'assigned_room_type']
Xcat = raw.drop(columns=['is_canceled'] + LEAKS)                          # string columns kept for slicing later
X = Xcat.apply(lambda c: c.astype('category').cat.codes if c.dtype == object else c).fillna(-1)   # integer codes: SHAP & MAPIE need numeric input
y = raw['is_canceled']
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=835)
GBM = dict(learning_rate=0.05, max_iter=2000, early_stopping=True, validation_fraction=0.15, n_iter_no_change=40, random_state=835)

## 1. Cross-validation that matches production
Plain `KFold` assumes rows are exchangeable. When the same customer (or agent, or company) appears many times, the model partly memorizes them and the score is optimistic. `GroupKFold` holds out whole groups. `TimeSeriesSplit` holds out the future.

In [ ]:
gbm = HistGradientBoostingClassifier(**GBM)
by_agent = X_tr['agent'].astype(int).values                                                      # bookings through the same travel agent look alike
by_week = (X_tr['arrival_date_year'] * 100 + X_tr['arrival_date_week_number']).values             # hold out whole weeks: closer to "score next month's bookings"
print(f"{(X_tr['agent'] == -1).mean():.0%} of bookings have no agent — GroupKFold must put that one giant group in a single fold\n")
for name, cv, g in [('KFold', KFold(5, shuffle=True, random_state=835), None), ('StratifiedKFold', StratifiedKFold(5, shuffle=True, random_state=835), None),
                    ('GroupKFold (by agent)', GroupKFold(5), by_agent), ('GroupKFold (by arrival week)', GroupKFold(5), by_week)]:
    s = cross_val_score(gbm, X_tr, y_tr, cv=cv, groups=g, scoring='roc_auc')
    print(f'{name:28s} AUC {s.mean():.4f} ± {s.std():.4f}')

Read the **±** before the mean. The agent-grouped score collapses *and* its spread balloons — mostly because the "no agent" lump (one seventh of the data) lands in a single fold, so that fold trains on a different population. That is a lopsided split, not proof the model fails on new agents. Grouping by arrival week gives a small, stable drop, and that is the number to report if the business will score *future* bookings — which it will. Choosing the splitter is a modeling decision — make it before the model, and check that the groups you hold out are not one giant lump.

## 2. Hyperparameter search: random, then Optuna
Grid search is exhaustive and slow; random search finds a near-best point in a tenth of the budget; Optuna learns from each trial. Tune on the training folds; judge once on the test set.

In [ ]:
space = {'learning_rate': [0.02, 0.05, 0.1, 0.2], 'max_leaf_nodes': [15, 31, 63, 127], 'min_samples_leaf': [10, 20, 50, 100], 'l2_regularization': [0, 0.1, 1.0, 5.0]}
rs = RandomizedSearchCV(HistGradientBoostingClassifier(**{**GBM, 'max_iter': 600}), space, n_iter=8, cv=StratifiedKFold(3, shuffle=True, random_state=835), scoring='roc_auc', random_state=835, n_jobs=-1).fit(X_tr, y_tr)
print('random search best:', rs.best_params_, f'CV AUC {rs.best_score_:.4f}')

In [ ]:
# OPTIONAL — pip install optuna
import optuna; optuna.logging.set_verbosity(optuna.logging.WARNING)
def objective(trial):
    params = dict(learning_rate=trial.suggest_float('learning_rate', 0.02, 0.3, log=True), max_leaf_nodes=trial.suggest_int('max_leaf_nodes', 8, 128, log=True),
                  min_samples_leaf=trial.suggest_int('min_samples_leaf', 5, 200, log=True), l2_regularization=trial.suggest_float('l2_regularization', 1e-3, 10, log=True))
    m = HistGradientBoostingClassifier(**{**GBM, **params, 'max_iter': 600})
    return cross_val_score(m, X_tr, y_tr, cv=StratifiedKFold(3, shuffle=True, random_state=835), scoring='roc_auc').mean()
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=835)); study.optimize(objective, n_trials=12)
print('Optuna best:', {k: round(v, 3) for k, v in study.best_params.items()}, f'CV AUC {study.best_value:.4f}')
best = HistGradientBoostingClassifier(**{**GBM, **study.best_params}).fit(X_tr, y_tr)
print(f'test AUC (touched once): {roc_auc_score(y_te, best.predict_proba(X_te)[:, 1]):.4f}')

In [ ]:
if 'best' not in dir(): best = HistGradientBoostingClassifier(**GBM).fit(X_tr, y_tr)     # fallback if the Optuna cell was skipped
p_te = best.predict_proba(X_te)[:, 1]; print(f'model in use: test AUC {roc_auc_score(y_te, p_te):.4f}')

## 3. SHAP — one booking, then the whole book
Shapley values split a prediction fairly among the features. `TreeExplainer` computes them exactly for boosted trees. The **waterfall** answers *why this booking?*; the **beeswarm** shows the global picture.

In [ ]:
# OPTIONAL — pip install shap
import shap
Xs = X_te.sample(1500, random_state=835)
explainer = shap.TreeExplainer(best); sv = explainer(Xs)
if sv.values.ndim == 3: sv = sv[:, :, 1]                                                     # some versions return per-class
print('SHAP values computed for', sv.values.shape[0], 'bookings ×', sv.values.shape[1], 'features (integer-coded categories)')
i = int(np.argsort(-best.predict_proba(Xs)[:, 1])[0])
shap.plots.waterfall(sv[i], max_display=10)
print(f"booking {Xs.index[i]}: predicted cancel prob {best.predict_proba(Xs.iloc[[i]])[0, 1]:.2f}  (base rate {y_tr.mean():.2f})")

In [ ]:
# OPTIONAL — needs the SHAP cell above
shap.plots.beeswarm(sv, max_display=12)
imp = pd.Series(np.abs(sv.values).mean(axis=0), index=Xs.columns).sort_values(ascending=False)
print('mean |SHAP| — the honest global importance:'); print(imp.head(8).round(4))

Say it to an executive: *"For this booking, the non-refundable deposit and the 200-day lead time push cancellation risk up by X points; being a repeat guest pulls it down."* SHAP explains the **model**, not the world — it attributes, it does not intervene.

## 4. Partial dependence + ICE

In [ ]:
from sklearn.inspection import PartialDependenceDisplay
Xs = X_te.sample(1500, random_state=835)                      # own sample, so this cell does not depend on the optional SHAP cell
fig, ax = plt.subplots(figsize=(10, 3.6))
PartialDependenceDisplay.from_estimator(best, Xs, ['lead_time', 'adr'], kind='both', subsample=200, ax=ax, random_state=835)
plt.tight_layout(); plt.show()

## 5. Conformal prediction — intervals that keep their promise
Use a calibration set's residuals to wrap *any* model in sets with guaranteed coverage. 90% means 90%, no distributional assumptions. A set of {cancel, stay} is the model saying "send this one to a human."

In [ ]:
# OPTIONAL — pip install mapie   (API shown is MAPIE ≥ 1.0)
from mapie.classification import SplitConformalClassifier
X_fit, X_cal, y_fit, y_cal = train_test_split(X_tr, y_tr, test_size=0.3, stratify=y_tr, random_state=835)
clf = HistGradientBoostingClassifier(**GBM).fit(X_fit, y_fit)
cp = SplitConformalClassifier(estimator=clf, confidence_level=0.9, prefit=True, conformity_score='lac'); cp.conformalize(X_cal, y_cal)   # numeric input required
y_pred, y_sets = cp.predict_set(X_te)
sets = y_sets[:, :, 0]; n_labels = sets.sum(axis=1)
coverage = sets[np.arange(len(y_te)), y_te.values].mean()
print(f'target 90% → empirical coverage {coverage:.3f}   singletons {np.mean(n_labels == 1):.1%}   ambiguous {{both}} {np.mean(n_labels == 2):.1%}   empty {np.mean(n_labels == 0):.1%}')

In [ ]:
# Regression version on Ames prices: 90% intervals in dollars
from mapie.regression import SplitConformalRegressor
from sklearn.datasets import fetch_openml
from sklearn.ensemble import HistGradientBoostingRegressor
ames = fetch_openml('house_prices', as_frame=True); A = ames.data.drop(columns=['Id']); ya = np.log1p(ames.target)
A = A.apply(lambda c: c.astype('category').cat.codes if c.dtype == object else c).fillna(-1)
A_tr, A_te, ya_tr, ya_te = train_test_split(A, ya, test_size=0.2, random_state=835); A_fit, A_cal, ya_fit, ya_cal = train_test_split(A_tr, ya_tr, test_size=0.3, random_state=835)
reg = HistGradientBoostingRegressor(random_state=835).fit(A_fit, ya_fit)
cr = SplitConformalRegressor(estimator=reg, confidence_level=0.9, prefit=True); cr.conformalize(A_cal, ya_cal)
pt, iv = cr.predict_interval(A_te); lo, hi = np.expm1(iv[:, 0, 0]), np.expm1(iv[:, 1, 0]); actual = np.expm1(ya_te.values)
print(f'coverage {np.mean((actual >= lo) & (actual <= hi)):.3f}   median interval width ${np.median(hi - lo):,.0f}   e.g. house 0: ${lo[0]:,.0f} – ${hi[0]:,.0f} (actual ${actual[0]:,.0f})')

## 6. Fairness slices
Equal accuracy can hide unequal harm. Compute false-negative and false-positive rates by group and look for gaps.

In [ ]:
pred = (p_te >= 0.5).astype(int)
def rates(mask):
    yt, yp = y_te.values[mask], pred[mask]
    return pd.Series({'n': mask.sum(), 'cancel rate': yt.mean(), 'FNR': ((yt == 1) & (yp == 0)).sum() / max((yt == 1).sum(), 1), 'FPR': ((yt == 0) & (yp == 1)).sum() / max((yt == 0).sum(), 1)})
for col in ['hotel', 'customer_type', 'market_segment']:
    labels = Xcat.loc[X_te.index, col]
    print(f'\n— by {col} —'); print(pd.DataFrame({g: rates((labels == g).values) for g in labels.unique() if (labels == g).sum() > 200}).T.round(3))

## 7. The model card (template)
```
MODEL CARD — Hotel cancellation risk v1
Intended use:      rank bookings by cancellation risk for overbooking & deposit policy. Not for pricing individuals.
Training data:     84k bookings, 2 Portuguese hotels, 2015–2017; leaks removed (reservation_status*, assigned_room_type).
Model:             HistGradientBoosting, early-stopped; threshold 0.5 (revisit with Session 6 cost matrix).
Performance:       test AUC 0.96; FNR/FPR by hotel & segment above; conformal 90% sets → ~X% ambiguous.
Known failures:    groups with < 200 bookings; future weeks score slightly lower than random CV; post-2017 drift unknown.
Monitoring:        input drift weekly (lead_time, deposit_type mix); AUC monthly once outcomes arrive; retrain quarterly.
Owner / review:    <name> · next review <date>
```

## 8. Your turn
1. **GroupKFold by `company`.** Is the grouped score lower again? Which grouping is closest to how the model will be used?
2. **Three waterfalls.** One confident cancel, one confident stay, one with a {both} conformal set. Two sentences each for a hotel manager.
3. **Fairness fix.** Pick the group with the highest FNR. Would a group-specific threshold help, and what would it cost the other groups?

In [ ]:
# Your turn — work here

## What we learned tonight
- **Match the splitter to production** — GroupKFold when units repeat, TimeSeriesSplit when time matters. Tune on training folds; touch the test set once.
- **SHAP answers "why this prediction?"** locally and "what drives the model?" globally — attribution, not intervention.
- **Conformal prediction gives guaranteed coverage for any model;** sliced error rates make "is it fair?" a table; the model card is where the answers live.

**HW4** due Mon Nov 23 — see the session page.